#  Step 1: Data Augmentation with Teacher LLM

This notebook generates high-quality, varied cricket commentary using a 'Teacher' model.
Use `pipeline.py augment` for the canonical end-to-end workflow.
**Stability Update:** We are using **Gemma 2 2B** in native `bfloat16`. 
- No `bitsandbytes` (incompatible with Nightly Pytorch)
- No `AutoAWQ` (deprecated/slow)
- **Fast & Stable** on RTX 5080.

In [ ]:
# Install core dependencies only
%pip install -qU transformers accelerate torch tqdm datasets

In [ ]:
import json
import torch
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# --- CONFIG ---
DATA_DIR = "data" 
OUTPUT_FILE = "data/rich_commentary_train.jsonl"

# Switching to 2B model. It fits easily in VRAM (4GB) in half-precision.
TEACHER_MODEL_ID = "google/gemma-2-2b-it" 
NUM_SAMPLES = 1000 

print(f" Initializing Teacher Model: {TEACHER_MODEL_ID}...")

🚀 Initializing Teacher Model: google/gemma-2-2b-it...


In [ ]:
# Load Model (bfloat16)
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150
)

print("Teacher Model Loaded (bfloat16)")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Device set to use cuda:0


✅ Teacher Model Loaded (bfloat16)


In [9]:
def generate_rich_commentary(delivery_json):
    batter = delivery_json['batter']
    bowler = delivery_json['bowler']
    runs = delivery_json['runs']['batter']
    extras = delivery_json.get('extras', {})
    wicket = delivery_json.get('wickets', [])
    
    prompt = f"""
    You are an expert cricket commentator like Harsha Bhogle or Ravi Shastri.
    Write a single sentence of exciting ball-by-ball commentary for the following delivery:
    
    Bowler: {bowler}
    Batter: {batter}
    Runs Scored: {runs}
    Extras: {extras}
    Wicket: {wicket}
    
    Commentary:
    """
    
    messages = [{"role": "user", "content": prompt}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    outputs = pipe(prompt_formatted, do_sample=True, temperature=0.7, top_p=0.9)
    generated_text = outputs[0]["generated_text"]
    
    # Extract response
    response = generated_text.split("<start_of_turn>model\n")[-1].strip()
    return response


In [ ]:
import glob
import random
import os

files = glob.glob(f"{DATA_DIR}/*.json")
print(f" Found {len(files)} match files.")

if not files:
    print(f" WARNING: No files found in '{DATA_DIR}'. Check your directory path! Current CWD: {os.getcwd()}")
    # Trying absolute path fallback if common download path
    abs_path = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/data"
    if os.path.exists(abs_path):
        print(f"Trying absolute path: {abs_path}")
        files = glob.glob(f"{abs_path}/*.json")
        print(f" Found {len(files)} match files at absolute path.")
    
if files:
    random.shuffle(files)
    selected_files = files[:100]

    rich_data = []
    count = 0

    print(" Generating Commentary...")
    for file in tqdm(selected_files):
        if count >= NUM_SAMPLES: break
        with open(file, 'r') as f: match = json.load(f)
        
        inning = random.choice(match['innings'])
        over = random.choice(inning['overs'])
        delivery = random.choice(over['deliveries'])
        
        try:
            commentary = generate_rich_commentary(delivery)
            rich_data.append({
                "instruction": "Write exciting cricket commentary for this ball.",
                "input": json.dumps(delivery),
                "output": commentary
            })
            count += 1
        except Exception as e:
            print(f" Error: {e}")

    # Ensure Directory Exists
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w") as f:
        for entry in rich_data:
            f.write(json.dumps(entry) + "\n")
    print(f"\nSaved {len(rich_data)} high-quality samples to {OUTPUT_FILE}")
else:
    print("❌ No files to process. Please fix DATA_DIR.")

📂 Found 0 match files.
⚠️ WARNING: No files found in 'data'. Check your directory path! Current CWD: /home/pragn
🔄 Trying absolute path: /mnt/c/Users/pragn/Downloads/Gemma_finetune/data
📂 Found 1169 match files at absolute path.
🎙️ Generating Commentary...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:11<00:00,  1.31s/it]


✅ Saved 100 high-quality samples to data/rich_commentary_train.jsonl
